# Week 1 — Smoke test

Sanity-check the artifacts produced by `uv run aegis backtest week1`:

- `data/processed/daily_panel_week1.parquet` — Module A output (Day 3)
- `data/processed/factor_mom_12_1_week1.parquet` — Module C output (Day 5)
- `data/ledger.sqlite` — Module B research ledger (Day 4 writes, Day 6 populates)

**Prerequisites:** run `uv run aegis backtest week1` first. Install plotting deps with `uv sync --extra notebook`.

This notebook is a smoke test, not a research report. It confirms the pipeline landed something coherent; the §11 success criteria (HAC t, DSR, FF6 α) don't apply until Module E (Week 13-15).

## 1. Load artifacts

In [ ]:
from pathlib import Path

import pandas as pd
import matplotlib.pyplot as plt
from sqlalchemy import select

from aegis.ledger.schema import Artifact, Candidate, Experiment
from aegis.ledger.store import open_ledger

REPO = Path('..').resolve()
PANEL = pd.read_parquet(REPO / 'data' / 'processed' / 'daily_panel_week1.parquet')
FACTOR = pd.read_parquet(REPO / 'data' / 'processed' / 'factor_mom_12_1_week1.parquet')
LEDGER = REPO / 'data' / 'ledger.sqlite'

print(f'panel:  {len(PANEL):>6,} rows   {PANEL["ticker"].nunique()} tickers   {PANEL["date"].min().date()} -> {PANEL["date"].max().date()}')
print(f'factor: {len(FACTOR):>6,} rows   {FACTOR["valid_flag"].sum():>5,} valid')

## 2. Ledger summary

Every `aegis backtest week1` run appends one experiment + one candidate + two artifacts.

In [ ]:
with open_ledger(LEDGER) as session:
    exps = session.execute(select(Experiment)).scalars().all()
    cands = session.execute(select(Candidate)).scalars().all()
    arts = session.execute(select(Artifact)).scalars().all()

    print(f'experiments: {len(exps)}')
    for e in exps[-3:]:
        print(f'  {e.experiment_id[:8]}  {e.name:24s}  config={e.config_hash[:8]}  git={e.git_sha[:10]}  @ {e.created_at.date()}')

    print(f'\ncandidates: {len(cands)}')
    for c in cands[-3:]:
        print(f'  {c.candidate_id[:8]}  {c.candidate_name:12s}  status={c.status:10s}  {c.formula_string}')

    print(f'\nartifacts:  {len(arts)}')
    for a in arts[-4:]:
        print(f'  {a.artifact_type:6s}  cksum={a.checksum[:12]}  {a.path}')

## 3. Eligibility over time

Count of eligible tickers per date. Ramps up as the 252-day history requirement is satisfied.

In [ ]:
per_date = PANEL.groupby('date')['eligible_flag'].sum()

fig, ax = plt.subplots(figsize=(10, 3.5))
per_date.plot(ax=ax, linewidth=1.2)
ax.set_title('Eligible tickers per trading day')
ax.set_ylabel('eligible count')
ax.set_xlabel('')
ax.grid(alpha=0.3)
fig.tight_layout()

## 4. Factor distribution on the final date

In [ ]:
last_date = FACTOR['date'].max()
cross_section = FACTOR[(FACTOR['date'] == last_date) & FACTOR['valid_flag']]

print(f'{last_date.date()} cross-section: {len(cross_section)} valid tickers')
print(cross_section[['ticker', 'raw_value', 'winsorized_value', 'zscore_value']]
      .sort_values('zscore_value', ascending=False)
      .to_string(index=False, float_format='%+.4f'))

## 5. AAPL raw 12-1 momentum over time

In [ ]:
aapl = FACTOR[(FACTOR['ticker'] == 'AAPL') & FACTOR['valid_flag']].set_index('date').sort_index()

fig, ax = plt.subplots(figsize=(10, 3.5))
aapl['raw_value'].plot(ax=ax, label='raw_value', linewidth=1.2)
ax.axhline(0, color='k', linewidth=0.5, alpha=0.5)
ax.set_title('AAPL 12-1 momentum (log(P[t-21] / P[t-252]))')
ax.set_ylabel('log return')
ax.grid(alpha=0.3)
ax.legend()
fig.tight_layout()

## 6. Sanity: per-date z-score mean ≈ 0

If the pipeline is correct, every per-date cross-section of z-scores has mean ≈ 0 and std ≈ 1 (population std). Any drift here means the transform layer broke.

In [ ]:
stats = (
    FACTOR[FACTOR['valid_flag']]
    .groupby('date')['zscore_value']
    .agg(['mean', 'std'])
)

print(f'per-date zscore mean: max|mean| = {stats["mean"].abs().max():.2e}')
print(f'per-date zscore std:  mean(std) = {stats["std"].mean():.4f}  (expect ~ 1.0 for populations >= 2)')
print()
print('first 5 dates:')
print(stats.head().round(6).to_string())